# Phase 2 — LangChain LCEL, Chains, and Streaming

## 1. LCEL Pipe Composition
The pipe operator | chains runnables together. Each runnable passes its output as input to the next.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models.fake_chat_models import FakeListChatModel

model = FakeListChatModel(responses=["Paris is the capital of France."])
prompt = ChatPromptTemplate.from_template("What is the capital of {country}?")
chain = prompt | model | StrOutputParser()
result = chain.invoke({"country": "France"})
print(result)

## 2. Parallel Chains with RunnableParallel
Run multiple chains concurrently. Each branch processes the same input independently.

In [ ]:
from langchain_core.runnables import RunnableParallel

model1 = FakeListChatModel(responses=["Short answer"])
model2 = FakeListChatModel(responses=["Detailed explanation with more context"])
parallel = RunnableParallel(
    brief=prompt | model1 | StrOutputParser(),
    detailed=prompt | model2 | StrOutputParser(),
)
result = parallel.invoke({"country": "France"})
print("Brief:", result["brief"])
print("Detailed:", result["detailed"])

## 3. Structured Output
model.with_structured_output() always beats manual JSON parsing — it uses native tool calling.

In [ ]:
from pydantic import BaseModel
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class FactCheck(BaseModel):
    claim: str
    is_supported: bool
    confidence: float


print("FactCheck schema:")
import json

print(json.dumps(FactCheck.model_json_schema(), indent=2))

## 4. Streaming Tokens
astream() yields chunks as they arrive. Essential for real-time UX.

In [ ]:
import asyncio


async def stream_demo():
    model = FakeListChatModel(responses=["Token by token streaming output"])
    chain = ChatPromptTemplate.from_template("{query}") | model | StrOutputParser()
    print("Streaming: ", end="")
    async for chunk in chain.astream({"query": "explain RAG"}):
        print(chunk, end="", flush=True)
    print()


asyncio.run(stream_demo())

## 5. Tool-Calling Agent
create_react_agent from langgraph.prebuilt builds a full ReAct agent over any set of tools.

In [ ]:
from langchain_core.tools import tool


@tool
def search_docs(query: str) -> str:
    "Search documentation for a query."
    return f"Results for: {query}"


@tool
def get_time() -> str:
    "Get the current time."
    import datetime

    return datetime.datetime.now().isoformat()


print("Tools defined:")
for t in [search_docs, get_time]:
    print(f"  {t.name}: {t.description}")

## Key Takeaways
- LCEL | pipes any Runnable together
- RunnableParallel fans out concurrently (wall-clock = slowest branch)
- with_structured_output() is always more reliable than manual JSON parsing
- astream() / astream_events() enable real-time streaming UX
- create_react_agent wraps tools into a full ReAct loop with checkpointing